# 01 — Data cleaning

Columbia Speed Dating Experiment (Fisman et al., 2006). Raw file: `data/raw/Speed_Dating_Data.csv` (encoding `cp1252`). **Never modify the raw CSV.**

## Team policies

1. **Zero→NA:** Partner and self 1–10 ratings outside [1, 10] (including **0**) are set to missing. No imputation.
2. **Preference rescale:** Complete six-attribute groups (`*1_1`, `*4_1`, `*2_1`, `*1_2`, `*7_2`, `pf_o_*`) are rescaled to sum to **100** per row. Incomplete or invalid groups become all-missing for that group.
3. **Incomplete sign-up prefs (`*1_1`):** Rows without a valid sextet stay missing (~1.4% of rows); not dropped.
4. **No row deletion** and no global z-scoring in the shared clean file (standardise ratings in model notebooks).

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.cleaning import run_cleaning
from src.constants import ATTRS, CLEAN_PATH, STATED, STATED_LOOKBACK

df = run_cleaning()
print(f"Shape: {df.shape}")
print(f"Wrote: {ROOT / CLEAN_PATH}")

Shape: (8378, 195)
Wrote: /Users/ktap8/Documents/work/University/Fourth Year/NUS/Data Driven Marketing/BrainOverLooksBT4211/data/processed/clean_unscaled.csv


In [2]:
stated = df[STATED]
complete_1_1 = stated.notna().all(axis=1).sum()
print(f"Complete *1_1 rows: {complete_1_1} / {len(df)} ({100*complete_1_1/len(df):.1f}%)")
print(f"Incomplete *1_1 rows: {len(df) - complete_1_1}")

if complete_1_1:
    print("*1_1 sum (complete rows):", stated.dropna(how="any").sum(axis=1).describe())

Complete *1_1 rows: 8257 / 8378 (98.6%)
Incomplete *1_1 rows: 121
*1_1 sum (complete rows): count    8257.0
mean      100.0
std         0.0
min       100.0
25%       100.0
50%       100.0
75%       100.0
max       100.0
dtype: Float64


In [3]:
lb = df[STATED_LOOKBACK]
complete_7_2 = lb.notna().all(axis=1).sum()
print(f"Complete *7_2 rows: {complete_7_2}")
if complete_7_2:
    print("*7_2 sum:", lb.dropna(how="any").sum(axis=1).describe())

Complete *7_2 rows: 1955
*7_2 sum: count    1955.0
mean      100.0
std         0.0
min       100.0
25%       100.0
50%       100.0
75%       100.0
max       100.0
dtype: Float64


In [4]:
print("Partner rating completeness:")
for c in ATTRS:
    print(f"  {c}: {100*df[c].notna().mean():.1f}%")

Partner rating completeness:
  attr: 97.5%
  sinc: 96.6%
  intel: 96.4%
  fun: 95.7%
  amb: 91.4%
  shar: 86.6%
